In [ ]:
import os
import glob
import json
import numpy as np
import pandas as pd
from pathlib import Path
from PNW_cmap import PNW_cmap
import matplotlib.pyplot as plt
from vip_slap2_analysis.utils.utils import save_figure
from vip_slap2_analysis.io.session_registry import VIPSessionRegistry
from vip_slap2_analysis.glutamate.summary import GlutamateSummary
from vip_slap2_analysis.utils.utils import normalize
from scipy.signal import find_peaks

import seaborn as sns
sns.set_style('white')
params = {'legend.fontsize': 'x-large',
         'axes.labelsize': 'xx-large',
         'axes.titlesize':'xx-large',
         'xtick.labelsize':'xx-large',
         'ytick.labelsize':'xx-large'}
plt.rcParams.update(params)

from IPython.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

In [ ]:
%load_ext autoreload
%autoreload 2

%matplotlib notebook

In [ ]:
savepath = r'C:\Users\andrew.shelton\Dropbox\allen institute\Documents\Presentations\OPhys\Data_Club\April2026\figures'

In [ ]:
target_mice = [
    803496,
    804730,804733,810196,
    809047,803121,
    826033,838410,834788
]

registry = VIPSessionRegistry.from_basepath(
    r'\\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics'
)

process_df = registry.sessions(
    subject_ids=target_mice,
    exclude_session_types=["expression_check", "volume_imaging"],
    paradigms=["change_detection_passive"],
)

assets = [registry.resolve_assets(row) for _, row in process_df.iterrows()]

print(f"Loaded {len(assets)} session assets")

In [ ]:
seq_sums = []
seq_pars = []
seq_pos = []
for asset in assets:
    try:
        print(asset.session_id)
        derived_dir = asset.derived_dir / 'glutamate' /'glutamate_analysis'
        seq_sum = pd.read_csv(os.path.join(derived_dir, 'sequence_summary_table.csv'))
        seq_sum['dmd1_depth'] = [asset.metadata['dmd1_depth']]*len(seq_sum)
        seq_sum['dmd2_depth'] = [asset.metadata['dmd2_depth']]*len(seq_sum)
        seq_par = pd.read_parquet(derived_dir / 'sequence_per_image_table.parquet')
        seq_pos_ = pd.read_parquet(derived_dir / 'sequence_position_table.parquet')
        seq_sums.append(seq_sum)

        seq_pars.append(seq_par)
        seq_pos.append(seq_pos_)
    except:
        pass

seq_summary = pd.concat(seq_sums)
seq_per_image = pd.concat(seq_pars)
seq_position = pd.concat(seq_pos)

In [ ]:
seq_per_image

In [ ]:
seq_paths = [glob.glob(os.path.join(asset.derived_dir,'**','glutamate_sequence_df.npz'),recursive=True)[0] for asset in assets]

In [ ]:
im_colors = [
    '#c5cae9', '#ffcdd2', '#c8e6c9', '#ffe0b2',
    '#e1bee7', '#d7ccc8',
    '#9fd3f2']

In [ ]:
# Facilitation example
mouse = '810196'
example_sess = f'{mouse}_2025-07-28_19-59-05'
example_dmd = 2
example_synapse = '0015'

example_datapath = [path for path in seq_paths if mouse in path][1]

seq_data = np.load(example_datapath,allow_pickle=True)['data'][0]

keys = list(seq_data[f'DMD{example_dmd}']['image_identity'].keys())

syn_im_df = seq_per_image[(seq_per_image['session_id']==example_sess)
                           &(seq_per_image['synapse_id']==f'DMD{example_dmd}_syn{example_synapse}')]

pref_image =  syn_im_df[syn_im_df['image_rank_within_synapse']==1]['stimulus_name'].values[0]

syn_pos_df = seq_position[(seq_position['session_id']==example_sess)
                           &(seq_position['synapse_id']==f'DMD{example_dmd}_syn{example_synapse}')]

In [ ]:
syn = int(example_synapse.split('0')[-1])
im = keys.index(pref_image)

mean_im_resps = (
    seq_data[f'DMD{example_dmd}']['image_identity'][pref_image]['repeated']['mean']
    .transpose(1, 0, 2)[syn][:]
)

resp_means = [np.mean(resp) for resp in mean_im_resps]
peaks = [resp[find_peaks(resp, distance=120)[0][0]] for resp in mean_im_resps]

shape = mean_im_resps.shape
concat_traces = mean_im_resps.reshape(shape[0] * shape[1])

fig, ax = plt.subplots(figsize=(6, 3))
ax.tick_params(axis='x', which='major', reset=True, top=False, labelsize=12)
ax.tick_params(axis='y', which='major', reset=True, right=False, labelsize=12)

mean = concat_traces
mean_rolling = pd.DataFrame(concat_traces).rolling(20, min_periods=1).mean()

time = np.linspace(0, len(mean) / 200, len(mean))
# ax.plot(time, mean, color='k')

ax.set_xlabel('Time (s)')
ax.set_ylabel('\u0394F', rotation=0, labelpad=15)

fs = 200
flash_start = 50 / fs      # 0.25 s
flash_dur = 50 / fs        # 0.25 s
gray_dur = 100 / fs        # 0.5 s
cycle_dur = flash_dur + gray_dur

n_flashes = shape[0]

for spine in ['left', 'right', 'top', 'bottom']:
    ax.spines[spine].set_linewidth(2)

windows = []

for i in range(n_flashes):
    start = flash_start + i * cycle_dur
    end = start + flash_dur
    ax.axvspan(start, end, alpha=0.4, color=im_colors[im])

    window_t = np.mean([start, end])
    windows.append(window_t)

windows = np.asarray(windows)
peaks = np.asarray(peaks)

# plot measured peaks
ax.plot(
    windows,
    resp_means,
    color='k',
    marker='o',
    lw=2,
    markerfacecolor=im_colors[im],
    label='Peak response'
)

# # -------------------------------------------------------
# # Slope fitting
# # -------------------------------------------------------
use_quantiles = False   # set True to fit on quantile-binned peaks
n_quantiles = 4         # only used if use_quantiles = True

if not use_quantiles:
    # simple linear fit to all peak points
    slope, intercept = np.polyfit(windows, resp_means, 1)
    fit_y = slope * windows + intercept

    ax.plot(
        windows,
        fit_y,
        color='red',
        lw=2.5,
        label=f'Linear fit (slope = {slope:.4f})'
    )

else:
    # quantile/bin-based fit across the ordered presentations
    fit_df = pd.DataFrame({
        'window': windows,
        'peak': peaks,
        'presentation_idx': np.arange(len(windows))
    })

    fit_df['quantile'] = pd.qcut(
        fit_df['presentation_idx'],
        q=n_quantiles,
        labels=False,
        duplicates='drop'
    )

    quant_summary = (
        fit_df.groupby('quantile', observed=True)
        .agg(
            window=('window', 'mean'),
            peak=('peak', 'mean')
        )
        .reset_index(drop=True)
    )

    qx = quant_summary['window'].to_numpy()
    qy = quant_summary['peak'].to_numpy()

    slope, intercept = np.polyfit(qx, qy, 1)
    fit_y = slope * qx + intercept

    # optional: show quantile means
    ax.plot(
        qx,
        qy,
        color='red',
        marker='o',
        lw=0,
        ms=7,
        label='Quantile means'
    )

    ax.plot(
        qx,
        fit_y,
        color='red',
        lw=2.5,
        label=f'Quantile fit (slope = {slope:.4f})'
    )

# ax.set_title('Synaptic Facilitation')
# ax.legend(frameon=False)
fig.tight_layout()

filen = 'Synaptic_Facilitation_example_slope'
save_figure(fig, os.path.join(savepath, filen), formats=['.pdf', '.png'], dpi=300)

In [ ]:
# Facilitation example
mouse = '803121'
example_sess = f'{mouse}_2025-11-01_19-00-21'
example_dmd = 2
example_synapse = '0065'

example_datapath = [path for path in seq_paths if mouse in path][3]

seq_data = np.load(example_datapath,allow_pickle=True)['data'][0]

keys = list(seq_data[f'DMD{example_dmd}']['image_identity'].keys())

syn_im_df = seq_per_image[(seq_per_image['session_id']==example_sess)
                           &(seq_per_image['synapse_id']==f'DMD{example_dmd}_syn{example_synapse}')]

pref_image =  syn_im_df[syn_im_df['image_rank_within_synapse']==1]['stimulus_name'].values[0]

syn_pos_df = seq_position[(seq_position['session_id']==example_sess)
                           &(seq_position['synapse_id']==f'DMD{example_dmd}_syn{example_synapse}')]

In [ ]:
syn = int(example_synapse.split('0')[-1])
im = keys.index(pref_image)

mean_im_resps = (
    seq_data[f'DMD{example_dmd}']['image_identity'][pref_image]['repeated']['mean']
    .transpose(1, 0, 2)[syn][:15]
)

resp_means = [np.mean(resp) for resp in mean_im_resps]
peaks = [resp[find_peaks(resp, distance=120)[0][0]] for resp in mean_im_resps]

shape = mean_im_resps.shape
concat_traces = mean_im_resps.reshape(shape[0] * shape[1])

fig, ax = plt.subplots(figsize=(6, 3))
ax.tick_params(axis='x', which='major', reset=True, top=False, labelsize=12)
ax.tick_params(axis='y', which='major', reset=True, right=False, labelsize=12)

mean = concat_traces
mean_rolling = pd.DataFrame(concat_traces).rolling(20, min_periods=1).mean()

time = np.linspace(0, len(mean) / 200, len(mean))
ax.plot(time, mean, color='k')

ax.set_xlabel('Time (s)')
ax.set_ylabel('\u0394F', rotation=0, labelpad=25)

fs = 200
flash_start = 50 / fs      # 0.25 s
flash_dur = 50 / fs        # 0.25 s
gray_dur = 100 / fs        # 0.5 s
cycle_dur = flash_dur + gray_dur

n_flashes = shape[0]

for spine in ['left', 'right', 'top', 'bottom']:
    ax.spines[spine].set_linewidth(2)

windows = []

for i in range(n_flashes):
    start = flash_start + i * cycle_dur
    end = start + flash_dur
    ax.axvspan(start, end, alpha=0.4, color=im_colors[im])

    window_t = np.mean([start, end])
    windows.append(window_t)

windows = np.asarray(windows)
peaks = np.asarray(peaks)

# # plot measured peaks
# ax.plot(
#     windows,
#     resp_means,
#     color='k',
#     marker='o',
#     lw=2,
#     markerfacecolor=im_colors[im],
#     label='Peak response'
# )

# # # -------------------------------------------------------
# # # Slope fitting
# # # -------------------------------------------------------
# use_quantiles = False   # set True to fit on quantile-binned peaks
# n_quantiles = 4         # only used if use_quantiles = True

# if not use_quantiles:
#     # simple linear fit to all peak points
#     slope, intercept = np.polyfit(windows, resp_means, 1)
#     fit_y = slope * windows + intercept

#     ax.plot(
#         windows,
#         fit_y,
#         color='red',
#         lw=2.5,
#         label=f'Linear fit (slope = {slope:.4f})'
#     )

# else:
#     # quantile/bin-based fit across the ordered presentations
#     fit_df = pd.DataFrame({
#         'window': windows,
#         'peak': peaks,
#         'presentation_idx': np.arange(len(windows))
#     })

#     fit_df['quantile'] = pd.qcut(
#         fit_df['presentation_idx'],
#         q=n_quantiles,
#         labels=False,
#         duplicates='drop'
#     )

#     quant_summary = (
#         fit_df.groupby('quantile', observed=True)
#         .agg(
#             window=('window', 'mean'),
#             peak=('peak', 'mean')
#         )
#         .reset_index(drop=True)
#     )

#     qx = quant_summary['window'].to_numpy()
#     qy = quant_summary['peak'].to_numpy()

#     slope, intercept = np.polyfit(qx, qy, 1)
#     fit_y = slope * qx + intercept

#     # optional: show quantile means
#     ax.plot(
#         qx,
#         qy,
#         color='red',
#         marker='o',
#         lw=0,
#         ms=7,
#         label='Quantile means'
#     )

#     ax.plot(
#         qx,
#         fit_y,
#         color='red',
#         lw=2.5,
#         label=f'Quantile fit (slope = {slope:.4f})'
#     )

# # ax.set_title('Synaptic Adaptation')
# ax.legend(frameon=False)
fig.tight_layout()

filen = 'Synaptic_Adaptation_example'
save_figure(fig, os.path.join(savepath, filen), formats=['.pdf', '.png'], dpi=300)